<div style="display:flex;gap:14px;align-items:center;flex-wrap:wrap;
 font-family:'Segoe UI',system-ui,sans-serif;font-size:13px;padding:10px 2px;
 border-bottom:2px solid #1B7A43;margin-bottom:4px;">
 <a href="https://colab.research.google.com/github/STG17-Africa/stg17-workshop/blob/main/notebooks/day4/D4_NTL_GEE_FR_open.ipynb" target="_blank"><img
  src="https://colab.research.google.com/assets/colab-badge.svg" alt="Ouvrir dans Colab"></a>
 <span style="color:#6B7B75;">Jour 4 · piste ouverte</span>
 <span style="flex:1;"></span>
 <a href="./D4_NTL_GEE_EN_open.ipynb" style="color:#1B7A43;font-weight:600;
  text-decoration:none;">🌐 English</a>
 <a href="./D4_NTL_GEE_FR.ipynb" style="color:#1B7A43;font-weight:600;
  text-decoration:none;">⇄ piste guidée</a>
</div>

<!-- Lumières nocturnes sur Google Earth Engine - aucun téléchargement · STG17 workshop · AfDB / STATAFRIC -->
<!-- GENERATED FILE — edit notebooks/_masters/d4_ntl_gee.master.ipynb instead. -->


<div style="background:linear-gradient(135deg,#0B2545 0%,#1B7A43 100%);
 border-radius:18px;padding:32px 38px;font-family:'Segoe UI',system-ui,sans-serif;">
 <div style="color:#F2A900;font-size:12.5px;letter-spacing:3px;font-weight:700;
  text-transform:uppercase;">Jour 4 · variante Earth Engine · s'exécute dans Colab</div>
 <div style="color:#fff;font-size:2em;font-weight:800;margin:10px 0 8px;line-height:1.15;">
  Les lumières nocturnes sans rien télécharger</div>
 <div style="color:#dbe7e0;font-size:1.05em;line-height:1.55;max-width:900px;">
  La même analyse que <code>D4_NTL_Collect_Explore</code>, mais en laissant les pixels où ils sont.
  Vous envoyez une expression ; Google l'exécute sur sa copie de l'archive ; un tableau revient.
  Une statistique pays-année qui coûte 800 Mo de téléchargement en local coûte ici quelques kilo-octets.
 </div>
 <div style="color:#F2A900;font-size:13px;margin-top:14px;font-weight:600;">
  Livrable : le même panel national, plus une comparaison documentée avec la voie locale.</div>
</div>


### Quand utiliser ce carnet plutôt que le carnet local

Utilisez Earth Engine quand la bande passante est la contrainte — ce qui, sur la
plupart des réseaux institutionnels de la région, est le cas. Utilisez la voie
locale quand la contrainte est la reproductibilité sur une machine hors ligne, ou
quand vous avez besoin d'un produit que Google n'héberge pas.

| | Local (rasterio) | Earth Engine |
|---|---|---|
| Réseau | aucun requis | requis |
| Qui détient les pixels | vous | Google |
| Reproductible hors ligne | oui | non |
| Produits disponibles | tout ce que publie la NASA | ce que Google a intégré |
| Passe à l'échelle avec | votre disque et votre RAM | le cluster de Google |
| Dépendance de long terme | aucune | envers un service commercial |

Cette dernière ligne n'est pas une note de bas de page. Un office national de
statistique qui bâtit un indicateur de production sur l'offre gratuite d'un
service commercial contracte une dépendance qu'il ne maîtrise pas — la question
de souveraineté du Jour 1 après-midi, sous forme concrète. La réponse honnête est
généralement : **prototyper sur Earth Engine, produire sur son infrastructure**,
et ce carnet, avec son jumeau local, permet exactement cela.


---
## Prérequis et autorisation

Earth Engine exige un compte **gratuit** rattaché à un projet Google Cloud.
L'inscription prend quelques minutes sur
[code.earthengine.google.com/register](https://code.earthengine.google.com/register)
et figure dans la liste préalable à l'atelier précisément pour cette raison.

Le premier `initialise()` ouvre une fenêtre de navigateur, une seule fois.
Ensuite, les identifiants sont mis en cache.


In [ ]:
# --- Prérequis de la variante Earth Engine -----------------------------
REQUIREMENTS = {
    "ee":         "earthengine-api>=0.1.380",
    "geemap":     "geemap>=0.30",
    "pandas":     "pandas>=2.0",
    "matplotlib": "matplotlib>=3.7",
}

import subprocess
import sys

try:
    import stg17
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "stg17 @ git+https://github.com/STG17-Africa/stg17-workshop"], check=False)
    import stg17

from stg17 import setup, countries, theme, ui
from stg17.i18n import T

S = setup(REQUIREMENTS, lang="FR")

In [ ]:
# L'identifiant de votre projet Cloud - depuis la page d'inscription Earth Engine.
GEE_PROJECT = None   # EN: e.g. "ee-yourname". None lets Earth Engine pick a default. | FR: ex. "ee-votrenom". None laisse Earth Engine choisir par défaut.

from stg17 import ntl_gee

ee = ntl_gee.initialise(project=GEE_PROJECT)

---
## Étape 1 — Votre pays

La même variable unique que dans le carnet local. Rien d'autre ne change.


In [ ]:
# ===========================================================================
# LA SEULE LIGNE À MODIFIER
# ===========================================================================
COUNTRY_ISO3 = "CIV"

YEARS         = list(range(2014, 2024))   # EN: EOG annual VNL covers 2012-2023 | FR: le VNL annuel EOG couvre 2012-2023
ADM_LEVEL     = 1
LIT_THRESHOLD = 0.5
SCALE_M       = 500       # EN: reduction resolution; raise to 1000 for a faster large country | FR: résolution de réduction ; passez à 1000 pour un grand pays plus rapide

C = countries.get(COUNTRY_ISO3)
OUT = S.outputs(C.iso3, "d4_ntl_gee")

nation = ntl_gee.country_geometry(ee, C, level=0)
zones = ntl_gee.country_geometry(ee, C, level=ADM_LEVEL)

n_zones = zones.size().getInfo()
print(f"{C.name('fr')} ({C.iso3})")
print(T(f"  ADM{ADM_LEVEL} units found in FAO GAUL: {n_zones}",
        f"  Unités ADM{ADM_LEVEL} trouvées dans FAO GAUL : {n_zones}"))
print(T(f"  Reduction scale: {SCALE_M} m    Years: {YEARS[0]}-{YEARS[-1]}",
        f"  Échelle de réduction : {SCALE_M} m    Années : {YEARS[0]}-{YEARS[-1]}"))

if n_zones == 0:
    print(T("No units returned. GAUL spells some country names differently from ISO - "
            "check stg17.ntl_gee._GAUL_ALIASES, or upload your own boundaries as an asset.",
            "Aucune unité retournée. GAUL orthographie certains noms de pays différemment de l'ISO - "
            "vérifiez stg17.ntl_gee._GAUL_ALIASES, ou téléversez vos propres frontières comme asset."))

<div style="border:1px solid #F2A900;border-left:6px solid #F2A900;background:#FEF9EC;
 padding:13px 17px;border-radius:0 9px 9px 0;margin:14px 0;font-family:'Segoe UI',system-ui,sans-serif;">
<b style="color:#F2A900;font-size:11.5px;letter-spacing:1.6px;">▲ ATTENTION — FAO GAUL DATE DE 2015</b><br>
<span style="color:#33403A;font-size:14.3px;line-height:1.58;">
Les frontières intégrées à Earth Engine sont celles de l'édition FAO GAUL 2015. Elles sont
antérieures à plusieurs réorganisations administratives sur le continent, et elles appellent
encore l'Eswatini « Swaziland ». Cela convient pour un prototype. Pour toute publication,
téléversez votre fichier de frontières national comme asset Earth Engine et passez-le via
<code>asset=</code> — la fonction le prend en charge, et le Jour 5 explique la procédure.
</span></div>


---
## Étape 2 — Observer une année, interactivement

`geemap` affiche une image Earth Engine comme une couche de tuiles vivante.
Déplacez et zoomez : les tuiles sont calculées à la demande, donc explorer un
pays entier ne coûte pas plus que d'explorer un district.

Notez la palette — c'est la même rampe `stg17_night` que celle du carnet local,
de sorte qu'une carte produite ici et une carte produite là-bas sont directement
comparables.


In [ ]:
# À FAIRE: Construisez une carte interactive pour l'année la plus récente avec ntl_gee.map_year()
...

---
## Étape 3 — Le panel : année × unité administrative

C'est le même tableau que celui construit par le carnet local, calculé côté
serveur. Ce qui revient par le réseau est de quelques centaines de lignes de
nombres.

Observez le chronomètre. Sur une connexion normale, c'est plus rapide que le
téléchargement d'un seul granule, et cela couvre dix années.


In [ ]:
# À FAIRE: Appelez ntl_gee.zonal_panel() sur YEARS et chronométrez
...

---
## Étape 4 — La série nationale

Deux panneaux : la Somme des lumières nationale indexée sur sa première année, et
la part du territoire au-dessus du seuil d'éclairement. Ensemble, ils séparent
« plus de lumière aux mêmes endroits » de « la lumière atteint de nouveaux
endroits » — une distinction capitale pour une lecture en termes
d'électrification, et qu'une série unique masque.


In [ ]:
# À FAIRE: Agrégez le panel au niveau national et tracez les deux séries
...

---
## Étape 5 — Comparer les deux chaînes

C'est l'étape qui justifie l'exécution de ce carnet même si vous avez déjà fait
le carnet local.

Chargez le panel produit par votre carnet local et placez les deux séries
nationales côte à côte. **Elles ne seront pas identiques.** Le VNL annuel de
l'EOG et le VNP46A4 de la NASA utilisent un compositage, une élimination des
valeurs aberrantes et un masquage différents. Un écart de quelques pour cent est
attendu et sain ; un écart d'un facteur deux signifie que l'une des deux
exécutions a un problème qu'il vaut la peine de trouver.

Ce qu'un office de statistique en retient : le nombre dépend de la chaîne, donc
la chaîne fait partie des métadonnées. Publier « Somme des lumières = 4,2
millions » sans nommer le produit et sa version n'est pas une statistique
reproductible.


In [ ]:
# À FAIRE: Chargez le panel local s'il existe et superposez les deux séries nationales indexées
...

---
## Étape 6 — Exporter et enregistrer

Deux sorties. Le panel en CSV, que le laboratoire de l'après-midi consomme. Et,
optionnellement, un GeoTIFF découpé exporté vers votre Drive — le pont pratique
entre les deux mondes : compositer sur le cluster de Google, puis poursuivre sous
rasterio sur un seul fichier national maniable plutôt que sur six tuiles de 10°.


In [ ]:
# À FAIRE: Enregistrez le panel en CSV avec un fichier de métadonnées, puis lancez éventuellement un export Drive
...

---
## Limites propres à cette voie

<div style="border:1px solid #F2A900;border-radius:10px;background:#FFFDF6;
 padding:16px 20px;margin:16px 0;font-family:'Segoe UI',system-ui,sans-serif;">
<ul style="color:#33403A;font-size:14px;line-height:1.7;margin:0;padding-left:22px;">
<li><b>Toutes les limites du carnet local s'appliquent encore</b> — statut d'indicateur indirect,
sensibilité au seuil, torchères, saturation, choix des frontières. Earth Engine change la façon
de calculer, pas la signification de la lumière.</li>
<li><b>Produit différent, chiffres différents.</b> Le VNL annuel EOG n'est pas le VNP46A4 de la
NASA. Nommez celui que vous avez utilisé dans chaque figure publiée.</li>
<li><b>Les frontières FAO GAUL 2015</b> sont la valeur par défaut ici et ne sont pas vos
frontières officielles.</li>
<li><b>L'échelle de réduction est un paramètre.</b> Réduire à 1000 m au lieu de 500 m est quatre
fois plus rapide et perd discrètement les petites agglomérations. Quel que soit votre choix,
consignez-le.</li>
<li><b>getInfo() est plafonné à 5000 entités.</b> Un pays comptant davantage d'unités ADM2 exige
un panel construit par groupes, ou exporté vers Drive sous forme de table.</li>
<li><b>Risque de continuité.</b> Ce résultat n'est reproductible que tant que Google sert la
collection à des conditions que vous pouvez accepter. Pour un indicateur statistique de
production, c'est une question de gouvernance à trancher avant l'adoption de la chaîne, pas
après.</li>
</ul></div>

<div style="background:linear-gradient(135deg,#0B2545,#1B7A43);border-radius:16px;
 padding:22px 30px;margin-top:22px;text-align:center;font-family:'Segoe UI',system-ui,sans-serif;">
 <div style="color:#F2A900;font-size:11.5px;letter-spacing:2.5px;font-weight:700;">
  DATA SCIENCE TOOLKIT · BAD / STATAFRIC</div>
 <div style="color:#fff;font-size:1.05em;margin-top:7px;font-weight:600;">
  Atelier STG17 · Plan d'action 2025-2030 · activités 4.2.1, 4.2.3 et 2.1.1</div></div>
